# 03 · Fine-tuning explicado y experimento Polar-solo vs preentrenado

## ¿Qué es el fine-tuning, en palabras sencillas?

Imagina que quieres reconocer a tus compañeros por su forma de caminar, pero solo los has visto caminar unos días. Es mucho más fácil si **antes** viste caminar a cientos de personas: ya sabes qué detalles importan (el balanceo de los brazos, el largo del paso) aunque esas personas no sean tus compañeros. Después solo te falta *afinar* lo que sabes con las pocas personas que sí te importan.

Con la red pasa lo mismo, en dos etapas:

1. **Preentrenamiento** con un dataset público (ECG-ID, muchas personas). La red aprende *cómo mirar* un ECG: dónde están el QRS, la onda T, cómo varían de persona a persona. Esas personas no son las nuestras; no importa.
2. **Fine-tuning** con las grabaciones del Polar. Primero cambiamos la *cabeza* (la parte final que solo sirve para entrenar; ahora hay otras personas y otro número de clases) y **congelamos** las capas tempranas para que no se desordenen (etapa A). Luego **descongelamos todo** y entrenamos con una tasa de aprendizaje mucho más baja, para ajustar sin borrar lo aprendido (etapa B).

**¿Por qué no solo entrenar con el Polar?** Con menos de 10 personas la red tiende a memorizar a esas personas en vez de aprender a distinguir en general. Este notebook mide si el preentrenamiento realmente ayuda.

In [ ]:
import sys, tempfile, dataclasses
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from src.config import Config, set_seed
from src.data import load_window_dir, split_polar
from src.synthetic import make_synthetic_table

set_seed(42)
polar_dir = ROOT / "data" / "processed" / "polar"
if list(polar_dir.glob("*.npz")):
    USING = "datos REALES del Polar H10"
    cfg = Config()
    table = load_window_dir(polar_dir)
    models_dir = ROOT / cfg.models_dir
else:
    USING = "datos SINTÉTICOS (aún no hay grabaciones en data/processed/polar)"
    cfg = dataclasses.replace(Config(), holdout_ids=("p4", "p5"), batch_size=32, patience=3,
                              polar_only_epochs=8, bootstrap_samples=200)
    table = make_synthetic_table(n_people=6, n_sessions=3, windows_per_session=40, seed=0)
    models_dir = Path(tempfile.mkdtemp())      # no ensuciamos models/ con pruebas
split = split_polar(table, cfg.holdout_ids)
from src.data import split_public
public_dir = ROOT / "data" / "processed" / "public"
if USING.startswith("datos REALES"):
    # Con Polar real NO se mezcla un público sintético: el resultado no significaría nada.
    if not list(public_dir.glob("*.npz")):
        raise FileNotFoundError(
            "Faltan los datos públicos. Ejecuta: python -m src.public_data download && "
            "python -m src.public_data build"
        )
    public = load_window_dir(public_dir)
    PUBLIC_USING = "ECG-ID real"
else:
    cfg = dataclasses.replace(cfg, pretrain_epochs=8, finetune_head_epochs=3, finetune_full_epochs=8)
    public = make_synthetic_table(n_people=20, n_sessions=2, windows_per_session=20, seed=7, prefix="pub_")
    PUBLIC_USING = "público SINTÉTICO (20 personas)"
public_train, public_val = split_public(public, cfg.pretrain_val_fraction, cfg.seed)
print("Preentrenamiento con:", PUBLIC_USING)
print("Usando:", USING)
print({"train": len(split.train), "val": len(split.val), "hold-out": len(split.holdout)}, "ventanas")

### Diagrama del proceso

In [ ]:
fig, ax = plt.subplots(figsize=(11, 2.6)); ax.axis("off")
steps = [("1. Preentrenar\n(dataset público,\nmuchas personas)", "#cfe8ff"),
         ("2A. Cabeza nueva\n+ capas tempranas\ncongeladas", "#ffe9b3"),
         ("2B. Todo descongelado\n(tasa de aprendizaje\nbaja)", "#ffd0d0"),
         ("3. Red final\n(huella de 128\nnúmeros)", "#d5f5d5")]
for i, (txt, color) in enumerate(steps):
    ax.text(i * 2.7 + 1.1, 0.5, txt, ha="center", va="center", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.6", fc=color, ec="gray"))
    if i < len(steps) - 1:
        ax.annotate("", xy=((i + 1) * 2.7 + 0.1, 0.5), xytext=(i * 2.7 + 2.1, 0.5),
                    arrowprops=dict(arrowstyle="->"))
ax.set_xlim(-0.3, 10.8); ax.set_ylim(0, 1); plt.show()

## Entrenamos los tres modelos con la MISMA partición de personas

In [ ]:
from src.train import pretrain, finetune, polar_only
import json
pre = pretrain(cfg, public_train, public_val, models_dir)
fin = finetune(cfg, split, pre, models_dir)
pol = polar_only(cfg, split, models_dir)
for name, p in (("preentrenado", pre), ("fine-tuned", fin), ("solo Polar", pol)):
    side = json.loads(p.with_suffix(".json").read_text(encoding="utf-8"))
    print(f"{name:14s} {p.name}  mejor EER de validación = {side['metrics']['best_val_eer']:.3f}")

## El experimento: EER de cada enfoque
El umbral de cada modelo se elige con su EER de **validación**; las filas *NO vistos (hold-out)* son personas que ni el preentrenamiento ni el fine-tuning vieron. Cada fila muestra cuántos pares genuinos/impostores y cuántas personas respaldan el número; con pocas personas el intervalo de confianza puede aparecer como *n/a* a propósito.

In [ ]:
from src.checkpoint import load_checkpoint
from src.evaluate import compare_models, make_model_embed_fn, raw_embed_fn
from IPython.display import Markdown, display
fns = {"Solo Polar": make_model_embed_fn(load_checkpoint(pol, cfg)),
       "Preentrenado + fine-tuning": make_model_embed_fn(load_checkpoint(fin, cfg)),
       "Señal cruda": raw_embed_fn}
display(Markdown(compare_models(fns, split, cfg)))

## Las huellas de personas no vistas: antes y después del fine-tuning
Mismas ventanas de hold-out proyectadas con PCA. Si el fine-tuning ayuda, los grupos por persona se ven más separados a la derecha que a la izquierda.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, ckpt) in zip(axes, (("Solo preentrenado", pre), ("Preentrenado + fine-tuning", fin), ("Solo Polar", pol))):
    emb = make_model_embed_fn(load_checkpoint(ckpt, cfg))(split.holdout.windows)
    proj = PCA(2).fit_transform(emb)
    for p in split.holdout.persons():
        m = split.holdout.participant_ids == p
        ax.scatter(proj[m, 0], proj[m, 1], s=12, label=p)
    ax.set_title(title)
axes[0].legend()
plt.tight_layout(); plt.show()

## Cómo leer los resultados
- Si *Preentrenado + fine-tuning* tiene EER menor que *Solo Polar* en **NO vistos**, el preentrenamiento ayuda y vale la pena su complejidad.
- Si son parecidos, la ventaja no está demostrada con estos datos; con más participantes puede cambiar. Reporta ambos resultados con honestidad.
- *Señal cruda* es la referencia sin entrenamiento: cualquier mejora sobre ella es lo que aporta la red.
- Con 2–3 personas de hold-out hay muy pocos pares impostores: los números son una **prueba de concepto**.